---

# Sprint 5 - calibration and operating point

Last task before the model is frozen. Two things, kept separate throughout:

- **Calibration** - do the probabilities mean what they say at screening prevalence?
- **Operating point** - where is the cut-off, and what does it cost?

**Pre-registered decision rule, fixed before any number below is computed.** Choose the
threshold that maximises breast-level sensitivity subject to a projected recall rate of no
more than **100 breasts per 1,000 screened**. Report the result whatever it is. If the
sensitivity at that cap is unusable, that is the finding, and the answer is to report it
rather than to raise the cap until the number looks acceptable.

Two things this notebook must get right or every number in it is wrong:

1. The development cohort is **case-control sampled** - all cancers, negatives subsampled
   at 12 per positive, per site. Sensitivity and specificity survive that unharmed;
   **recall rate and PPV do not**. Both are projected back to the true population using
   per-site weights reconstructed from the full RSNA release.
2. Calibration is fitted **with those same weights**, so it targets the screening
   population rather than the enriched sample.

The sealed holdout is not opened. Scoring uses the frozen wide 0.1-99.9 windowing and the
frozen `max_available_view_probability` fusion rule - no image is re-scored.

## K1.0 Load, and build the operational breast table

The fusion analysis used only breasts holding both views, because that is the only cohort
on which fusion methods can be compared. **Calibration is different**: it must cover every
breast the deployed system would score, so single-view breasts are included here under the
frozen rule.

In [59]:
import os, json, numpy as np, pandas as pd

RECALL_CAP_PER_1000 = 100.0        # pre-registered
REPORT_CAPS = [50.0, 100.0, 150.0]  # reported for context; only the cap above is frozen
NBOOT_K = 2000
RNG_K = np.random.default_rng(31337)

if 'OUT_DIR' not in globals():
    OUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
    os.makedirs(OUT_DIR, exist_ok=True)


def _find(fname):
    p = os.path.join(OUT_DIR, fname)
    if os.path.exists(p):
        return p
    for root in ('/kaggle/input', '.'):
        if os.path.isdir(root):
            for dp, _d, fs in os.walk(root):
                if fname in fs:
                    return os.path.join(dp, fname)
    return None


# --- image-level scores under the frozen windowing ---
if 'wres' in globals() and 'ensemble_wide' in globals()['wres'].columns:
    IMG = wres.copy(); _src = 'in-memory wres'
else:
    _p = _find('windowing_paired_scores_full.csv')
    if _p is None:
        raise SystemExit('windowing_paired_scores_full.csv not found.')
    IMG = pd.read_csv(_p); _src = _p

SC = [c[:-5] for c in IMG.columns if c.endswith('_wide')]
ENSC = 'ensemble' if 'ensemble' in SC else SC[-1]
print('scores from :', _src)
print('scorers     :', SC, '| primary:', ENSC)

# --- the frozen fusion rule ---
_ff = _find('frozen_fusion.json')
FUSION = json.load(open(_ff)) if _ff else {'method': 'max_available_view_probability'}
if FUSION.get('method') != 'max_available_view_probability':
    raise SystemExit('frozen_fusion.json says method=%r. Run the fusion patch first.'
                     % FUSION.get('method'))
print('fusion rule :', FUSION['method'])

# --- split manifest, and the holdout guard ---
_mp = _find('rsna_split_manifest.csv')
if _mp is None:
    raise SystemExit('rsna_split_manifest.csv not found - cannot prove the holdout is excluded.')
MAN = pd.read_csv(_mp)
DEV_P = set(MAN.loc[MAN.split == 'dev', 'patient_id'])
HOLD_P = set(MAN.loc[MAN.split == 'holdout', 'patient_id'])
if set(IMG.patient_id) & HOLD_P:
    raise SystemExit('HOLDOUT LEAK in the score table. Stop.')
IMG = IMG[IMG.patient_id.isin(DEV_P)].copy()
print('holdout leak: none  [PASS]   dev patients %d' % len(DEV_P))

# --- build breast-level scores under the frozen rule ---
IMG['view'] = IMG['view'].astype(str).str.upper().str.strip()
IMG['laterality'] = IMG['laterality'].astype(str).str.upper().str.strip()
IMG = IMG[IMG.view.isin(['CC', 'MLO']) & IMG.laterality.isin(['L', 'R'])].copy()
IMG['breast'] = IMG.patient_id.astype(str) + '_' + IMG.laterality

_val = {s: s + '_wide' for s in SC}
byview = (IMG.groupby(['patient_id', 'breast', 'laterality', 'view'], as_index=False)
             .agg(cancer=('cancer', 'max'),
                  **{v: (v, 'mean') for v in _val.values()}))          # mean within view
BRK = (byview.groupby(['patient_id', 'breast', 'laterality'], as_index=False)
             .agg(cancer=('cancer', 'max'), n_views=('view', 'nunique'),
                  **{v: (v, 'max') for v in _val.values()}))           # max across views


def _mode(s):
    m = s.mode()
    return m.iat[0] if len(m) else np.nan


_cov = [c for c in ['site_id', 'machine_id', 'density'] if c in IMG.columns]
_bc = IMG.groupby('breast').agg(**{c: (c, _mode) for c in _cov}, age=('age', 'median'))
BRK = BRK.merge(_bc, on='breast', how='left')
BRK['age_band'] = pd.cut(BRK.age, [0, 50, 60, 70, 200],
                         labels=['<50', '50-60', '60-70', '70+'], right=False)
BRK = BRK.rename(columns={v: k for k, v in _val.items()}).reset_index(drop=True)

print()
print('operational breast table: %d breasts, %d patients, %d cancer breasts'
      % (len(BRK), BRK.patient_id.nunique(), int(BRK.cancer.sum())))
print(BRK.n_views.value_counts().rename('breasts by views present').to_string())
print()
print('  (the fusion analysis used only the %d two-view breasts; calibration covers all of them)'
      % int((BRK.n_views == 2).sum()))

scores from : in-memory wres
scorers     : ['v11_dro', 'v8_resnet', 'ensemble'] | primary: ensemble
fusion rule : max_available_view_probability
holdout leak: none  [PASS]   dev patients 6214

operational breast table: 8355 breasts, 6211 patients, 343 cancer breasts
n_views
1    6650
2    1705

  (the fusion analysis used only the 1705 two-view breasts; calibration covers all of them)


## K1.1 True prevalence and sampling weights

The evaluation cohort kept every cancer and subsampled negatives at 12 per positive within
each site, so the sample is roughly 7.6% cancer while the screening population is under 1%
at breast level. Every rate that depends on the mix of positives and negatives - recall
rate, PPV - has to be projected back.

The weights are reconstructed from the **full RSNA release restricted to development
patients**, so they are counted rather than assumed. Each negative breast in the sample
stands for `true negatives / sampled negatives` negative breasts at its site; positives
carry weight 1 because none were dropped.

If the full release is unavailable the cell stops rather than silently using sample
prevalence, which would understate the recall rate by roughly an order of magnitude.

In [60]:
_full = None
if 'rsna' in globals():
    _full = globals()['rsna']
    _fsrc = 'in-memory rsna'
else:
    _p = _find('train.csv')
    if _p:
        _full = pd.read_csv(_p)
        _fsrc = _p
if _full is None:
    raise SystemExit('The full RSNA table (rsna / train.csv) is required to reconstruct '
                     'sampling weights. Without it, recall rate and PPV cannot be projected '
                     'and would be wrong by roughly 10x. Attach the competition data.')

_f = _full[_full.patient_id.isin(DEV_P)].copy()
_f['laterality'] = _f['laterality'].astype(str).str.upper().str.strip()
_f['breast'] = _f.patient_id.astype(str) + '_' + _f.laterality
TRUEBR = _f.groupby(['breast', 'site_id'], as_index=False).cancer.max()

t_pos = TRUEBR[TRUEBR.cancer == 1].groupby('site_id').size()
t_neg = TRUEBR[TRUEBR.cancer == 0].groupby('site_id').size()
s_pos = BRK[BRK.cancer == 1].groupby('site_id').size()
s_neg = BRK[BRK.cancer == 0].groupby('site_id').size()

W = pd.DataFrame({'true_pos': t_pos, 'true_neg': t_neg,
                  'samp_pos': s_pos, 'samp_neg': s_neg}).fillna(0)
W['w_pos'] = np.where(W.samp_pos > 0, W.true_pos / W.samp_pos.replace(0, np.nan), 1.0)
W['w_neg'] = np.where(W.samp_neg > 0, W.true_neg / W.samp_neg.replace(0, np.nan), 1.0)
W['true_prev'] = W.true_pos / (W.true_pos + W.true_neg)
print('per-site reconstruction (full release, development patients only), source: %s' % _fsrc)
print(W.round(4).to_string())

BRK['w'] = [W.loc[r.site_id, 'w_pos'] if r.cancer == 1 else W.loc[r.site_id, 'w_neg']
            for r in BRK.itertuples()]

TRUE_PREV = float(TRUEBR.cancer.mean())
SAMPLE_PREV = float(BRK.cancer.mean())
WEIGHTED_PREV = float((BRK.w * BRK.cancer).sum() / BRK.w.sum())
print()
print('breast-level prevalence, development patients')
print('  in the full release        : %.5f  (%d of %d breasts)'
      % (TRUE_PREV, int(TRUEBR.cancer.sum()), len(TRUEBR)))
print('  in the sampled cohort      : %.5f' % SAMPLE_PREV)
print('  after reweighting          : %.5f' % WEIGHTED_PREV)
assert abs(WEIGHTED_PREV - TRUE_PREV) < 0.01, 'weights do not reproduce the true prevalence'
print('  weights reproduce the population prevalence  [PASS]')

per-site reconstruction (full release, development patients only), source: in-memory rsna
         true_pos  true_neg  samp_pos  samp_neg  w_pos   w_neg  true_prev
site_id                                                                  
1             178      6352       178      4380    1.0  1.4502     0.0273
2             165      5733       165      3632    1.0  1.5785     0.0280

breast-level prevalence, development patients
  in the full release        : 0.02760  (343 of 12428 breasts)
  in the sampled cohort      : 0.04105
  after reweighting          : 0.02760
  weights reproduce the population prevalence  [PASS]


## K1.2 Calibration

Member temperatures were fitted on Mammo-Bench at 44.5% malignant. At under 1% the
probabilities are far too high, so a breast scored 0.62 is not a 62% chance of cancer -
it is nothing like it.

A two-parameter Platt scaling is fitted on the breast-level logit, **weighted** so the fit
targets the screening population rather than the enriched sample. Two parameters on 343
positive breasts is a modest fit; anything richer would overfit.

Reported before and after: weighted Brier score, weighted expected calibration error, and
a decile reliability table. Calibration changes what a probability *means*; it does not
change the ranking, so AUC is untouched by design and is reported as a check that nothing
was broken rather than as a result.

In [61]:
EPS = 1e-6


def _logit(p):
    p = np.clip(np.asarray(p, float), EPS, 1 - EPS)
    return np.log(p / (1 - p))


def _sig(z):
    return 1.0 / (1.0 + np.exp(-z))


def fit_platt(z, y, w, iters=200):
    """Weighted 2-parameter Platt scaling by IRLS. No sklearn dependency."""
    a, b = 1.0, 0.0
    for _ in range(iters):
        p = _sig(a * z + b)
        Wt = w * p * (1 - p)
        g = np.array([(w * (y - p) * z).sum(), (w * (y - p)).sum()])
        H = np.array([[(Wt * z * z).sum(), (Wt * z).sum()],
                      [(Wt * z).sum(), Wt.sum()]]) + np.eye(2) * 1e-9
        step = np.linalg.solve(H, g)
        a += step[0]; b += step[1]
        if np.max(np.abs(step)) < 1e-11:
            break
    return float(a), float(b)


def wbrier(y, p, w):
    return float((w * (p - y) ** 2).sum() / w.sum())


def wece(y, p, w, bins=10):
    edges = np.quantile(p, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    tot, e = w.sum(), 0.0
    for i in range(bins):
        m = (p > edges[i]) & (p <= edges[i + 1])
        if not m.any():
            continue
        ww = w[m].sum()
        e += ww / tot * abs((w[m] * p[m]).sum() / ww - (w[m] * y[m]).sum() / ww)
    return float(e)


y_k = BRK.cancer.to_numpy(float)
w_k = BRK.w.to_numpy(float)
CAL = {}
rel_rows = []
for s in SC:
    z = _logit(BRK[s].to_numpy(float))
    a, b = fit_platt(z, y_k, w_k)
    p_raw = BRK[s].to_numpy(float)
    p_cal = _sig(a * z + b)
    BRK[s + '_cal'] = p_cal
    CAL[s] = {'a': a, 'b': b,
              'brier_raw': wbrier(y_k, p_raw, w_k), 'brier_cal': wbrier(y_k, p_cal, w_k),
              'ece_raw': wece(y_k, p_raw, w_k), 'ece_cal': wece(y_k, p_cal, w_k),
              'mean_raw': float((w_k * p_raw).sum() / w_k.sum()),
              'mean_cal': float((w_k * p_cal).sum() / w_k.sum())}
    print('%-10s a=%.4f b=%+.4f | weighted Brier %.5f -> %.5f | ECE %.5f -> %.5f | '
          'mean predicted %.4f -> %.5f  (observed %.5f)'
          % (s, a, b, CAL[s]['brier_raw'], CAL[s]['brier_cal'],
             CAL[s]['ece_raw'], CAL[s]['ece_cal'],
             CAL[s]['mean_raw'], CAL[s]['mean_cal'], WEIGHTED_PREV))

    ed = np.quantile(p_cal, np.linspace(0, 1, 11)); ed[0], ed[-1] = -np.inf, np.inf
    for i in range(10):
        m = (p_cal > ed[i]) & (p_cal <= ed[i + 1])
        if not m.any():
            continue
        ww = w_k[m].sum()
        rel_rows.append(dict(scorer=s, decile=i + 1, n_breasts=int(m.sum()),
                             cancer_breasts=int(y_k[m].sum()),
                             mean_predicted=(w_k[m] * p_cal[m]).sum() / ww,
                             observed_rate=(w_k[m] * y_k[m]).sum() / ww))

REL = pd.DataFrame(rel_rows)
REL.to_csv(os.path.join(OUT_DIR, 'calibration_curve.csv'), index=False)
print()
print('=== reliability after calibration, %s ===' % ENSC)
print(REL[REL.scorer == ENSC].round(5).to_string(index=False))
print()
print('saved calibration_curve.csv')

v11_dro    a=1.2115 b=-4.7627 | weighted Brier 0.40131 -> 0.02490 | ECE 0.59272 -> 0.00461 | mean predicted 0.6203 -> 0.02760  (observed 0.02760)
v8_resnet  a=0.3415 b=-3.6271 | weighted Brier 0.30511 -> 0.02663 | ECE 0.45674 -> 0.00549 | mean predicted 0.4843 -> 0.02760  (observed 0.02760)
ensemble   a=0.8003 b=-4.2588 | weighted Brier 0.40077 -> 0.02617 | ECE 0.58420 -> 0.00446 | mean predicted 0.6118 -> 0.02760  (observed 0.02760)

=== reliability after calibration, ensemble ===
  scorer  decile  n_breasts  cancer_breasts  mean_predicted  observed_rate
ensemble       1        836               7         0.00568        0.00538
ensemble       2        835              17         0.00966        0.01333
ensemble       3        836              25         0.01295        0.02001
ensemble       4        835              19         0.01627        0.01529
ensemble       5        836              28         0.02015        0.02262
ensemble       6        835              31         0.02455    

## K1.3 Operating points

For every candidate threshold, projected to the screening population:

- **sensitivity** - share of cancer breasts flagged. Unaffected by the negative sampling.
- **specificity** - share of non-cancer breasts not flagged. Also unaffected.
- **recall rate per 1,000** - breasts flagged per 1,000 screened. Depends on the mix, so
  it is computed with the weights.
- **cancers found per 1,000** and **PPV** - likewise.
- **number needed to recall** - how many recalls buy one cancer.

The frozen threshold is the one that maximises sensitivity while keeping the projected
recall rate at or below the pre-registered cap.

In [62]:
def op_metrics(p, y, w, t):
    flag = p >= t
    wp, wn = w[y == 1], w[y == 0]
    tp = w[(y == 1) & flag].sum(); fn = wp.sum() - tp
    fp = w[(y == 0) & flag].sum(); tn = wn.sum() - fp
    tot = w.sum()
    sens = tp / wp.sum() if wp.sum() else np.nan
    spec = tn / wn.sum() if wn.sum() else np.nan
    rec = 1000.0 * (tp + fp) / tot
    det = 1000.0 * tp / tot
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    return dict(threshold=t, sensitivity=sens, specificity=spec,
                recall_per_1000=rec, cancers_found_per_1000=det, ppv=ppv,
                nnr=(1.0 / ppv if ppv and ppv > 0 else np.nan),
                n_flagged_sample=int(flag.sum()),
                cancer_breasts_found=int(((y == 1) & flag).sum()),
                cancer_breasts_missed=int(((y == 1) & ~flag).sum()))


p_ens = BRK[ENSC + '_cal'].to_numpy(float)
grid = np.unique(np.quantile(p_ens, np.linspace(0, 1, 2001)))
OPS = pd.DataFrame([op_metrics(p_ens, y_k, w_k, t) for t in grid])
OPS.to_csv(os.path.join(OUT_DIR, 'operating_points.csv'), index=False)

print('at the current model, projected to a population of %.4f%% cancer breasts:'
      % (100 * WEIGHTED_PREV))
print()
rows = []
for cap in sorted(set(REPORT_CAPS + [RECALL_CAP_PER_1000])):
    ok = OPS[OPS.recall_per_1000 <= cap]
    if len(ok) == 0:
        rows.append(dict(cap=cap, note='no threshold reaches this cap'))
        continue
    r = ok.loc[ok.sensitivity.idxmax()].to_dict()
    r['cap'] = cap
    rows.append(r)
CAPS = pd.DataFrame(rows)
print(CAPS[['cap', 'threshold', 'sensitivity', 'specificity', 'recall_per_1000',
            'cancers_found_per_1000', 'ppv', 'nnr', 'cancer_breasts_found',
            'cancer_breasts_missed']].round(5).to_string(index=False))

sel = CAPS[CAPS.cap == RECALL_CAP_PER_1000].iloc[0]
THRESH = float(sel.threshold)
print()
print('PRE-REGISTERED CAP %.0f per 1,000  ->  threshold %.6f' % (RECALL_CAP_PER_1000, THRESH))
print('  sensitivity %.4f   specificity %.4f   recall %.1f per 1,000   PPV %.4f   NNR %.0f'
      % (sel.sensitivity, sel.specificity, sel.recall_per_1000, sel.ppv, sel.nnr))
print('  finds %d of %d cancer breasts in the development sample'
      % (sel.cancer_breasts_found, int(y_k.sum())))
print()
print('For reference, the Sprint 4 operating point recalled 854 per 1,000.')

at the current model, projected to a population of 2.7599% cancer breasts:

  cap  threshold  sensitivity  specificity  recall_per_1000  cancers_found_per_1000     ppv      nnr  cancer_breasts_found  cancer_breasts_missed
 50.0    0.06652      0.18950      0.95419         49.77796                 5.23013 0.10507  9.51755                  65.0                  278.0
100.0    0.05397      0.29738      0.90562         99.98683                 8.20727 0.08208 12.18271                 102.0                  241.0
150.0    0.04670      0.38776      0.85720        149.56208                10.70164 0.07155 13.97562                 133.0                  210.0

PRE-REGISTERED CAP 100 per 1,000  ->  threshold 0.053967
  sensitivity 0.2974   specificity 0.9056   recall 100.0 per 1,000   PPV 0.0821   NNR 12
  finds 102 of 343 cancer breasts in the development sample

For reference, the Sprint 4 operating point recalled 854 per 1,000.


## K1.4 Confidence intervals at the frozen threshold

Patients are resampled; the threshold is held fixed at the value selected above, because it
is now a frozen constant rather than something re-chosen per resample. Re-selecting it
inside the bootstrap would answer a different question.

In [63]:
pats = BRK.patient_id.to_numpy()
uniq = np.unique(pats)
pidx = {p: np.where(pats == p)[0] for p in uniq}

boot = []
for _ in range(NBOOT_K):
    rows = np.concatenate([pidx[p] for p in RNG_K.choice(uniq, len(uniq), replace=True)])
    yy = y_k[rows]
    if yy.sum() == 0 or yy.sum() == len(yy):
        continue
    boot.append(op_metrics(p_ens[rows], yy, w_k[rows], THRESH))

BOOTK = pd.DataFrame(boot)
ci_rows = []
for m in ['sensitivity', 'specificity', 'recall_per_1000', 'cancers_found_per_1000',
          'ppv', 'nnr']:
    lo, hi = np.percentile(BOOTK[m].dropna(), [2.5, 97.5])
    ci_rows.append(dict(metric=m, estimate=float(sel[m]), lo=lo, hi=hi))
CI = pd.DataFrame(ci_rows)
print('%d resamples kept, unit: patient_id, threshold fixed at %.6f' % (len(BOOTK), THRESH))
print(CI.round(4).to_string(index=False))
CI.to_csv(os.path.join(OUT_DIR, 'operating_point_ci.csv'), index=False)
print()
print('saved operating_point_ci.csv')

2000 resamples kept, unit: patient_id, threshold fixed at 0.053967
                metric  estimate      lo       hi
           sensitivity    0.2974  0.2478   0.3478
           specificity    0.9056  0.8990   0.9125
       recall_per_1000   99.9868 93.3331 106.8443
cancers_found_per_1000    8.2073  6.6489   9.8587
                   ppv    0.0821  0.0667   0.0987
                   nnr   12.1827 10.1276  14.9874

saved operating_point_ci.csv


## K1.5 Subgroups at the frozen threshold

Sensitivity and recall rate by site, density, age band and machine, with counts. Strata
with fewer than ten cancer breasts get their counts printed and no rate, because a
sensitivity computed on four positives is one case away from a different answer.

In [64]:
MIN_POS_K = 10
sg = []
for c in ['site_id', 'density', 'age_band', 'machine_id']:
    if c not in BRK.columns:
        continue
    for k, g in BRK.dropna(subset=[c]).groupby(c, observed=True):
        yy = g.cancer.to_numpy(float); ww = g.w.to_numpy(float)
        pp = g[ENSC + '_cal'].to_numpy(float)
        npos = int(yy.sum())
        row = dict(stratum='%s %s' % (c.replace('_id', ''), k), n_breasts=len(g),
                   cancer_breasts=npos, noncancer_breasts=int(len(g) - npos))
        if npos >= MIN_POS_K:
            m = op_metrics(pp, yy, ww, THRESH)
            row.update({k2: m[k2] for k2 in ['sensitivity', 'specificity',
                                             'recall_per_1000', 'ppv',
                                             'cancer_breasts_found',
                                             'cancer_breasts_missed']})
            row['note'] = ''
        else:
            row['note'] = 'too few cancer breasts - no rate reported'
        sg.append(row)

SG = pd.DataFrame(sg)
SG.to_csv(os.path.join(OUT_DIR, 'calibration_subgroups.csv'), index=False)
print(SG.round(4).to_string(index=False))
print()
print('saved calibration_subgroups.csv')
_weak = SG[(SG.note == '') & (SG.sensitivity < float(sel.sensitivity) - 0.10)]
if len(_weak):
    print()
    print('Strata more than 10 points below the overall sensitivity of %.4f:' % sel.sensitivity)
    print(_weak[['stratum', 'cancer_breasts', 'sensitivity', 'recall_per_1000']]
          .round(4).to_string(index=False))

       stratum  n_breasts  cancer_breasts  noncancer_breasts  sensitivity  specificity  recall_per_1000    ppv  cancer_breasts_found  cancer_breasts_missed                                      note
        site 1       4558             178               4380       0.3371       0.8865         119.5656 0.0768                  60.0                  118.0                                          
        site 2       3797             165               3632       0.2545       0.9268          78.3101 0.0909                  42.0                  123.0                                          
     density A        468              12                456       0.4167       0.8289         175.4301 0.0423                   5.0                    7.0                                          
     density B       1986              80               1906       0.4375       0.8568         151.5091 0.0812                  35.0                   45.0                                          
     densi

## K1.6 Freeze

In [65]:
frozen_op = {
    'status': 'frozen operating point, development set only',
    'decision_rule': ('maximise breast-level sensitivity subject to a projected recall rate '
                      'of at most %.0f per 1,000 screened; pre-registered before any number '
                      'was computed' % RECALL_CAP_PER_1000),
    'unit': 'breast (patient_id + laterality)',
    'score': '%s, wide 0.1-99.9 windowing, max over available views, Platt-calibrated' % ENSC,
    'calibration': {'method': 'weighted 2-parameter Platt scaling on the breast-level logit',
                    'fitted_on': 'RSNA development split, sample weights reconstructed from '
                                 'the full release restricted to development patients',
                    'parameters': {s: {'a': CAL[s]['a'], 'b': CAL[s]['b']} for s in SC},
                    'weighted_brier': {s: {'raw': CAL[s]['brier_raw'],
                                           'calibrated': CAL[s]['brier_cal']} for s in SC},
                    'weighted_ece': {s: {'raw': CAL[s]['ece_raw'],
                                         'calibrated': CAL[s]['ece_cal']} for s in SC}},
    'threshold': THRESH,
    'projected_to_prevalence': WEIGHTED_PREV,
    'metrics': {r['metric']: {'estimate': r['estimate'], 'lo': r['lo'], 'hi': r['hi']}
                for _, r in CI.iterrows()},
    'cohort': {'breasts': int(len(BRK)), 'patients': int(BRK.patient_id.nunique()),
               'cancer_breasts': int(y_k.sum()),
               'two_view_breasts': int((BRK.n_views == 2).sum()),
               'single_view_breasts': int((BRK.n_views == 1).sum())},
    'alternatives': CAPS[['cap', 'threshold', 'sensitivity', 'recall_per_1000', 'ppv']]
                    .to_dict('records'),
    'sprint4_reference': {'recall_per_1000': 854, 'note': 'operating point tuned at 44.5% '
                                                          'prevalence'},
    'holdout': 'sealed and not opened; this threshold has never seen it',
    'not_done': ['holdout evaluation', 'retraining', 'any change to preprocessing or fusion'],
}
with open(os.path.join(OUT_DIR, 'frozen_operating_point.json'), 'w') as f:
    json.dump(frozen_op, f, indent=2, default=float)

L = []
A = L.append
A('# Sprint 5 - calibration and operating point')
A('')
A('**Pre-registered rule:** maximise breast-level sensitivity subject to a projected recall '
  'rate of at most %.0f per 1,000 screened. Fixed before any number below was computed.'
  % RECALL_CAP_PER_1000)
A('')
A('## Cohort')
A('')
A('- %d breasts from %d patients, %d cancer breasts. %d hold both views, %d hold one.'
  % (len(BRK), BRK.patient_id.nunique(), int(y_k.sum()),
     int((BRK.n_views == 2).sum()), int((BRK.n_views == 1).sum())))
A('- RSNA development split only. The sealed holdout was not opened.')
A('- Scores are the frozen wide 0.1-99.9 windowing pass, combined by the frozen '
  '`max_available_view_probability` rule. No image was re-scored.')
A('')
A('## Why the raw rates could not be used directly')
A('')
A('The cohort keeps every cancer and subsamples negatives at 12 per positive, so it is '
  '%.2f%% cancer at breast level while the screening population is %.3f%%. Sensitivity and '
  'specificity are unaffected by that, but recall rate and PPV are not, and both are '
  'projected using per-site weights reconstructed by counting the full release restricted '
  'to development patients.' % (100 * SAMPLE_PREV, 100 * TRUE_PREV))
A('')
A('## Calibration')
A('')
A('| model | a | b | weighted Brier, raw -> calibrated | weighted ECE, raw -> calibrated |')
A('|---|---|---|---|---|')
for s in SC:
    A('| `%s` | %.4f | %+.4f | %.5f -> %.5f | %.5f -> %.5f |'
      % (s, CAL[s]['a'], CAL[s]['b'], CAL[s]['brier_raw'], CAL[s]['brier_cal'],
         CAL[s]['ece_raw'], CAL[s]['ece_cal']))
A('')
A('Mean predicted probability for `%s` moved from %.4f to %.5f against an observed rate of '
  '%.5f. The member temperatures were fitted at 44.5%% prevalence, so the uncalibrated '
  'numbers were never interpretable as probabilities in a screening population. '
  'Calibration is monotone, so ranking and AUC are unchanged by construction.'
  % (ENSC, CAL[ENSC]['mean_raw'], CAL[ENSC]['mean_cal'], WEIGHTED_PREV))
A('')
A('## Frozen operating point')
A('')
A('Threshold **%.6f** on the calibrated `%s` breast probability.' % (THRESH, ENSC))
A('')
A('| metric | estimate | 95% CI |')
A('|---|---|---|')
for _, r in CI.iterrows():
    A('| %s | %.4f | %.4f to %.4f |' % (r['metric'].replace('_', ' '),
                                        r['estimate'], r['lo'], r['hi']))
A('')
A('For comparison, the Sprint 4 operating point recalled 854 per 1,000 - it was tuned on a '
  '44.5% prevalence population and was never valid for screening.')
A('')
A('## Alternatives at other caps')
A('')
A('| recall cap per 1,000 | threshold | sensitivity | recall achieved | PPV |')
A('|---|---|---|---|---|')
for _, r in CAPS.iterrows():
    if 'threshold' not in r or pd.isna(r.get('threshold', np.nan)):
        A('| %.0f | - | - | - | not reachable |' % r['cap'])
    else:
        A('| %.0f | %.6f | %.4f | %.1f | %.4f |'
          % (r['cap'], r['threshold'], r['sensitivity'], r['recall_per_1000'], r['ppv']))
A('')
A('Only the %.0f per 1,000 row is frozen. The others are reported so the clinical cap can '
  'be renegotiated with evidence rather than guessed at.' % RECALL_CAP_PER_1000)
A('')
A('## Subgroups at the frozen threshold')
A('')
A('| stratum | breasts | cancer | sensitivity | recall per 1,000 | note |')
A('|---|---|---|---|---|---|')
for _, r in SG.iterrows():
    if r['note']:
        A('| %s | %d | %d | - | - | %s |'
          % (r['stratum'], r['n_breasts'], r['cancer_breasts'], r['note']))
    else:
        A('| %s | %d | %d | %.4f | %.1f | |'
          % (r['stratum'], r['n_breasts'], r['cancer_breasts'],
             r['sensitivity'], r['recall_per_1000']))
A('')
A('## Scope')
A('')
A('- The sealed holdout was not opened. This threshold has never seen it, which is what '
  'makes the eventual holdout evaluation worth reporting.')
A('- Nothing was retrained and neither preprocessing nor fusion was changed.')
A('- Calibration was fitted on development data. Its parameters are part of the frozen '
  'model and must travel with it.')
A('')
_rep = os.path.join(OUT_DIR, 'calibration_report.md')
with open(_rep, 'w') as f:
    f.write('\n'.join(L))

BRK.to_csv(os.path.join(OUT_DIR, 'breast_scores_dev.csv'), index=False)
print('saved frozen_operating_point.json')
print('saved calibration_report.md')
print('saved breast_scores_dev.csv')
print()
print('\n'.join(L[:40]))

saved frozen_operating_point.json
saved calibration_report.md
saved breast_scores_dev.csv

# Sprint 5 - calibration and operating point

**Pre-registered rule:** maximise breast-level sensitivity subject to a projected recall rate of at most 100 per 1,000 screened. Fixed before any number below was computed.

## Cohort

- 8355 breasts from 6211 patients, 343 cancer breasts. 1705 hold both views, 6650 hold one.
- RSNA development split only. The sealed holdout was not opened.
- Scores are the frozen wide 0.1-99.9 windowing pass, combined by the frozen `max_available_view_probability` rule. No image was re-scored.

## Why the raw rates could not be used directly

The cohort keeps every cancer and subsamples negatives at 12 per positive, so it is 4.11% cancer at breast level while the screening population is 2.760%. Sensitivity and specificity are unaffected by that, but recall rate and PPV are not, and both are projected using per-site weights reconstructed by counting the full release r

## K1.7 What was written

| file | contents |
|---|---|
| `breast_scores_dev.csv` | breast-level scores, raw and calibrated, with weights |
| `calibration_curve.csv` | decile reliability, weighted |
| `operating_points.csv` | the full threshold grid with projected rates |
| `operating_point_ci.csv` | patient bootstrap intervals at the frozen threshold |
| `calibration_subgroups.csv` | sensitivity and recall by site, density, age, machine |
| `frozen_operating_point.json` | the frozen threshold and calibration parameters |
| `calibration_report.md` | the written decision |

Download them before the session ends.

**The model is now frozen** - preprocessing, fusion, calibration and threshold. The sealed
holdout can be opened exactly once, after that freeze is packaged. Do not open it in this
notebook.